# Gold Layer - Part 2.B: Analytical Model

Dataset: French Road Safety Open Data, year 2024.

This notebook builds the Gold layer star schema from the Silver tables:
`fact_accidents`, `fact_vehicles`, `fact_users`, and four small dimension
tables (`dim_date`, `dim_location`, `dim_vehicle_category`, `dim_severity`).

In [1]:
import pandas as pd
from pathlib import Path

SILVER_DIR = Path("data/silver")
GOLD_DIR = Path("data/gold")

## Load the Silver tables

In [2]:
accidents = pd.read_csv(SILVER_DIR / "accidents_silver.csv", parse_dates=["accident_date"], low_memory=False)
vehicules = pd.read_csv(SILVER_DIR / "vehicules_silver.csv", low_memory=False)
usagers = pd.read_csv(SILVER_DIR / "usagers_silver.csv", low_memory=False)

print(accidents.shape, vehicules.shape, usagers.shape)

(54402, 44) (92678, 11) (125187, 17)


## Reference lookup labels

In [3]:
DEPARTMENT_NAMES = {
    "01": "Ain", "02": "Aisne", "03": "Allier", "04": "Alpes-de-Haute-Provence",
    "05": "Hautes-Alpes", "06": "Alpes-Maritimes", "07": "Ardeche", "08": "Ardennes",
    "09": "Ariege", "10": "Aube", "11": "Aude", "12": "Aveyron", "13": "Bouches-du-Rhone",
    "14": "Calvados", "15": "Cantal", "16": "Charente", "17": "Charente-Maritime",
    "18": "Cher", "19": "Correze", "2A": "Corse-du-Sud", "2B": "Haute-Corse",
    "21": "Cote-d\'Or", "22": "Cotes-d\'Armor", "23": "Creuse", "24": "Dordogne",
    "25": "Doubs", "26": "Drome", "27": "Eure", "28": "Eure-et-Loir", "29": "Finistere",
    "30": "Gard", "31": "Haute-Garonne", "32": "Gers", "33": "Gironde", "34": "Herault",
    "35": "Ille-et-Vilaine", "36": "Indre", "37": "Indre-et-Loire", "38": "Isere",
    "39": "Jura", "40": "Landes", "41": "Loir-et-Cher", "42": "Loire", "43": "Haute-Loire",
    "44": "Loire-Atlantique", "45": "Loiret", "46": "Lot", "47": "Lot-et-Garonne",
    "48": "Lozere", "49": "Maine-et-Loire", "50": "Manche", "51": "Marne",
    "52": "Haute-Marne", "53": "Mayenne", "54": "Meurthe-et-Moselle", "55": "Meuse",
    "56": "Morbihan", "57": "Moselle", "58": "Nievre", "59": "Nord", "60": "Oise",
    "61": "Orne", "62": "Pas-de-Calais", "63": "Puy-de-Dome", "64": "Pyrenees-Atlantiques",
    "65": "Hautes-Pyrenees", "66": "Pyrenees-Orientales", "67": "Bas-Rhin",
    "68": "Haut-Rhin", "69": "Rhone", "70": "Haute-Saone", "71": "Saone-et-Loire",
    "72": "Sarthe", "73": "Savoie", "74": "Haute-Savoie", "75": "Paris",
    "76": "Seine-Maritime", "77": "Seine-et-Marne", "78": "Yvelines", "79": "Deux-Sevres",
    "80": "Somme", "81": "Tarn", "82": "Tarn-et-Garonne", "83": "Var", "84": "Vaucluse",
    "85": "Vendee", "86": "Vienne", "87": "Haute-Vienne", "88": "Vosges", "89": "Yonne",
    "90": "Territoire de Belfort", "91": "Essonne", "92": "Hauts-de-Seine",
    "93": "Seine-Saint-Denis", "94": "Val-de-Marne", "95": "Val-d\'Oise",
    "971": "Guadeloupe", "972": "Martinique", "973": "Guyane", "974": "La Reunion",
    "975": "Saint-Pierre-et-Miquelon", "976": "Mayotte",
}

LUM_LABELS = {
    1: "Daylight", 2: "Dusk or dawn", 3: "Night without public lighting",
    4: "Night with public lighting off", 5: "Night with public lighting on",
}

ATM_LABELS = {
    1: "Normal", 2: "Light rain", 3: "Heavy rain", 4: "Snow/hail",
    5: "Fog/smoke", 6: "Strong wind/storm", 7: "Dazzling weather",
    8: "Overcast", 9: "Other",
}

COL_LABELS = {
    1: "Two vehicles - head-on", 2: "Two vehicles - rear-end",
    3: "Two vehicles - side", 4: "Three or more vehicles - chain",
    5: "Three or more vehicles - multiple collisions", 6: "Other collision",
    7: "No collision",
}

CATV_LABELS = {
    1: "Bicycle", 2: "Moped", 3: "Quad/light motorcycle", 7: "Passenger car",
    10: "Utility vehicle", 13: "Truck under 7.5t", 14: "Truck 7.5t or more",
    15: "Truck with trailer", 17: "Road tractor", 21: "Agricultural tractor",
    30: "Scooter", 31: "Motorcycle", 32: "Scooter under 50cm3",
    33: "Motorcycle 50-125cm3", 34: "Motorcycle over 125cm3", 36: "Light quad",
    37: "Bus", 38: "Coach", 39: "Train", 40: "Tramway", 41: "Trailer 3-wheeler",
    42: "Trailer more than 3 wheels", 43: "Quad more than 50cm3", 50: "EDPM with motor",
    60: "EDPM without motor", 80: "Personal mobility device with motor",
    99: "Other vehicle",
}

GRAV_LABELS = {1: "Unharmed", 2: "Killed", 3: "Hospitalized injured", 4: "Slightly injured"}
CATU_LABELS = {1: "Driver", 2: "Passenger", 3: "Pedestrian", 4: "Pedestrian on personal mobility device"}
SEXE_LABELS = {1: "Male", 2: "Female"}

## Dimension tables

In [4]:
dates = accidents["accident_date"].dropna().drop_duplicates().sort_values()
dim_date = pd.DataFrame({"accident_date": dates})
dim_date["year"] = dim_date["accident_date"].dt.year
dim_date["month"] = dim_date["accident_date"].dt.month
dim_date["month_name"] = dim_date["accident_date"].dt.month_name()
dim_date["day"] = dim_date["accident_date"].dt.day
dim_date["weekday"] = dim_date["accident_date"].dt.day_name()
dim_date["is_weekend"] = dim_date["accident_date"].dt.dayofweek >= 5

dim_date.head()

,accident_date,year,month,month_name,day,weekday,is_weekend
4419,2024-01-01,2024,1,January,1,Monday,False
6532,2024-01-02,2024,1,January,2,Tuesday,False
5046,2024-01-03,2024,1,January,3,Wednesday,False
8528,2024-01-04,2024,1,January,4,Thursday,False
6568,2024-01-05,2024,1,January,5,Friday,False


In [5]:
dep_codes = accidents["dep"].dropna().drop_duplicates()
dim_location = pd.DataFrame({"dep": dep_codes})
dim_location["dep_name"] = dim_location["dep"].map(DEPARTMENT_NAMES).fillna("Unknown")

dim_vehicle_category = pd.DataFrame(
    {"catv": list(CATV_LABELS.keys()), "catv_label": list(CATV_LABELS.values())}
)

dim_severity = pd.DataFrame({"grav": list(GRAV_LABELS.keys()), "grav_label": list(GRAV_LABELS.values())})

dim_location.head()

,dep,dep_name
0,70,Haute-Saone
1,21,Cote-d'Or
2,15,Cantal
3,14,Calvados
4,13,Bouches-du-Rhone


## Fact tables

In [6]:
fact_accidents = accidents.copy()
fact_accidents["lum_label"] = fact_accidents["lum"].map(LUM_LABELS)
fact_accidents["atm_label"] = fact_accidents["atm"].map(ATM_LABELS)
fact_accidents["col_label"] = fact_accidents["col"].map(COL_LABELS)

fact_accidents = fact_accidents[[
    "Num_Acc", "accident_date", "hour", "time_of_day", "weekday",
    "dep", "com", "lat", "long", "vma",
    "lum", "lum_label", "atm", "atm_label", "col", "col_label", "agg",
    "nb_vehicles", "nb_users", "nb_killed", "nb_hospitalized",
    "nb_slightly_injured", "nb_unharmed", "severity_index", "severity_category",
]]

fact_accidents.head()

,Num_Acc,accident_date,hour,time_of_day,weekday,dep,com,lat,long,vma,...,col_label,agg,nb_vehicles,nb_users,nb_killed,nb_hospitalized,nb_slightly_injured,nb_unharmed,severity_index,severity_category
0,202400000001,2024-03-25,7,Morning,Monday,70,70285,47.562770,6.758320,90,...,Two vehicles - head-on,1,2,2,0,1,0,1,2,Severe
1,202400000002,2024-03-20,15,Afternoon,Wednesday,21,21054,47.021090,4.837550,30,...,Other collision,2,1,2,0,1,0,1,2,Severe
2,202400000003,2024-03-22,19,Evening,Friday,15,15012,44.902384,2.496418,50,...,Other collision,1,1,5,0,1,3,1,5,Severe
3,202400000004,2024-03-24,17,Afternoon,Sunday,14,14118,49.191660,-0.398510,50,...,Two vehicles - side,2,2,1,0,0,1,0,1,Minor
4,202400000005,2024-03-25,19,Evening,Monday,13,13106,43.390000,5.350000,50,...,Three or more vehicles - multiple collisions,1,4,4,0,0,3,1,3,Minor


In [7]:
fact_vehicles = vehicules.copy()
fact_vehicles["catv_label"] = fact_vehicles["catv"].map(CATV_LABELS).fillna("Other vehicle")
fact_vehicles = fact_vehicles[["Num_Acc", "id_vehicule", "catv", "catv_label", "choc", "manv", "senc", "motor"]]

fact_vehicles.head()

,Num_Acc,id_vehicule,catv,catv_label,choc,manv,senc,motor
0,202400000001,155 781 758,7,Passenger car,1.0,13,1.0,1.0
1,202400000001,155 781 759,14,Truck 7.5t or more,2.0,21,2.0,1.0
2,202400000002,155 781 757,10,Utility vehicle,3.0,15,1.0,1.0
3,202400000003,155 781 756,3,Quad/light motorcycle,1.0,1,3.0,1.0
4,202400000004,155 781 754,34,Motorcycle over 125cm3,0.0,0,3.0,1.0


In [8]:
fact_users = usagers.copy()
fact_users["grav_label"] = fact_users["grav"].map(GRAV_LABELS)
fact_users["catu_label"] = fact_users["catu"].map(CATU_LABELS)
fact_users["sexe_label"] = fact_users["sexe"].map(SEXE_LABELS)
fact_users = fact_users[[
    "Num_Acc", "id_usager", "catu", "catu_label", "grav", "grav_label",
    "sexe", "sexe_label", "age", "trajet",
]]

fact_users.head()

,Num_Acc,id_usager,catu,catu_label,grav,grav_label,sexe,sexe_label,age,trajet
0,202400000001,203 988 581,1,Driver,3,Hospitalized injured,1.0,Male,21.0,2
1,202400000001,203 988 582,1,Driver,1,Unharmed,1.0,Male,27.0,4
2,202400000002,203 988 579,3,Pedestrian,3,Hospitalized injured,2.0,Female,97.0,5
3,202400000002,203 988 580,1,Driver,1,Unharmed,1.0,Male,37.0,4
4,202400000003,203 988 574,2,Passenger,4,Slightly injured,2.0,Female,17.0,5


## Save the Gold layer

In [9]:
GOLD_DIR.mkdir(parents=True, exist_ok=True)

dim_date.to_csv(GOLD_DIR / "dim_date.csv", index=False)
dim_location.to_csv(GOLD_DIR / "dim_location.csv", index=False)
dim_vehicle_category.to_csv(GOLD_DIR / "dim_vehicle_category.csv", index=False)
dim_severity.to_csv(GOLD_DIR / "dim_severity.csv", index=False)

fact_accidents.to_csv(GOLD_DIR / "fact_accidents.csv", index=False)
fact_vehicles.to_csv(GOLD_DIR / "fact_vehicles.csv", index=False)
fact_users.to_csv(GOLD_DIR / "fact_users.csv", index=False)

print("Gold layer written to data/gold/")

Gold layer written to data/gold/
